<a href="https://colab.research.google.com/github/abrizu/480-NN-Project-MRI/blob/demdet/Dementia_audio_model_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# code for generating the word embeddings of audio and transcripts using the fine0tuned bert and wav3vec models
# word embeddigns are saved to "combined_new_embedding.pt" file in drive for later use
import os
import torch
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2Model, BertTokenizer, BertModel

wav_folders = [
    "/content/pitt_data/Pitt_Data/Dementia/wav/cookie",
    "/content/pitt_data/Pitt_Data/Control/wav/cookie"
]

txt_folders = [
    "/content/pitt_data/Pitt_Data/Dementia/transcripts/cookie",
    "/content/pitt_data/Pitt_Data/Control/transcripts/cookie"
]

combined_out = "/content/drive/MyDrive/combined_new_embedding.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

wav_processor = Wav2Vec2Processor.from_pretrained("/content/drive/MyDrive/finetuned_wav2vec")
wav_model = Wav2Vec2Model.from_pretrained("/content/drive/MyDrive/finetuned_wav2vec").to(device)
wav_model.eval()

bert_tokenizer = BertTokenizer.from_pretrained("/content/drive/MyDrive/finetuned_bert")
bert_model = BertModel.from_pretrained("/content/drive/MyDrive/finetuned_bert").to(device)
bert_model.eval()

def get_wav_embedding(wav_path, sr=16000):
    audio, _ = librosa.load(wav_path, sr=sr)
    inputs = wav_processor(audio, sampling_rate=sr, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = wav_model(**inputs)
    emb = out.last_hidden_state.mean(dim=1)[0].cpu()
    return emb

def get_text_embedding(txt_path, max_length=512):
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    if len(text) == 0:
        return torch.zeros(768)
    inputs = bert_tokenizer(text, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = bert_model(**inputs)
    if out.pooler_output is not None:
        emb = out.pooler_output[0].cpu()
    else:
        emb = out.last_hidden_state.mean(dim=1)[0].cpu()
    return emb

def process_group(wav_folder, txt_folder, label, start_idx):
    combined = {}
    idx = start_idx
    wav_files = {os.path.splitext(f)[0]: f for f in sorted(os.listdir(wav_folder)) if f.lower().endswith(".wav")}
    txt_files = {os.path.splitext(f)[0]: f for f in sorted(os.listdir(txt_folder)) if f.lower().endswith(".txt")}
    common_bases = [b for b in sorted(wav_files.keys()) if b in txt_files]
    for base in common_bases:
        wav_path = os.path.join(wav_folder, wav_files[base])
        txt_path = os.path.join(txt_folder, txt_files[base])
        try:
            w_emb = get_wav_embedding(wav_path)
            t_emb = get_text_embedding(txt_path)
        except Exception as e:
            print(f"⚠️ Skipping {base} due to error: {e}")
            continue
        combined_emb = torch.cat([t_emb, w_emb], dim=0)
        combined[idx] = (combined_emb, torch.tensor(label, dtype=torch.long))
        print(f"Processed {base}  label={label}  idx={idx}")
        idx += 1
    return combined, idx

if __name__ == "__main__":
    all_combined = {}
    current_idx = 0
    dementia_wav = wav_folders[0]; dementia_txt = txt_folders[0]
    if os.path.exists(dementia_wav) and os.path.exists(dementia_txt):
        c, current_idx = process_group(dementia_wav, dementia_txt, label=1, start_idx=current_idx)
        all_combined.update(c)
    else:
        print("⚠️ Dementia folders missing.")
    control_wav = wav_folders[1]; control_txt = txt_folders[1]
    if os.path.exists(control_wav) and os.path.exists(control_txt):
        c, current_idx = process_group(control_wav, control_txt, label=0, start_idx=current_idx)
        all_combined.update(c)
    else:
        print("⚠️ Control folders missing.")
    torch.save(all_combined, combined_out)
    print(f"\n✅ Saved {len(all_combined)} combined embeddings to:\n{combined_out}")


Device: cuda
Processed 001-0  label=1  idx=0
Processed 001-2  label=1  idx=1
Processed 003-0  label=1  idx=2
Processed 005-0  label=1  idx=3
Processed 005-2  label=1  idx=4
Processed 007-1  label=1  idx=5
Processed 007-3  label=1  idx=6
Processed 010-0  label=1  idx=7
Processed 010-1  label=1  idx=8
Processed 010-2  label=1  idx=9
Processed 010-3  label=1  idx=10
Processed 010-4  label=1  idx=11
Processed 014-2  label=1  idx=12
Processed 016-0  label=1  idx=13
Processed 016-1  label=1  idx=14
Processed 016-3  label=1  idx=15
Processed 016-4  label=1  idx=16
Processed 018-0  label=1  idx=17
Processed 023-0  label=1  idx=18
Processed 023-2  label=1  idx=19
Processed 024-1  label=1  idx=20
Processed 024-2  label=1  idx=21
Processed 029-0  label=1  idx=22
Processed 029-1  label=1  idx=23
Processed 030-0  label=1  idx=24
Processed 030-1  label=1  idx=25
Processed 033-0  label=1  idx=26
Processed 033-1  label=1  idx=27
Processed 033-2  label=1  idx=28
Processed 033-3  label=1  idx=29
Process

In [ ]:
# code for loading pitt_data dataset in here and using it for fine-tuning the models
from google.colab import drive
drive.mount('/content/drive')

import os

# path to your file in google drive
src = "/content/drive/MyDrive/Pitt_Data.zip"
dst = "/content/Pitt_Data.zip"

# copy file to runtime
print("Copying file...")
os.system(f'cp "{src}" "{dst}"')

# verify file
print("\nFiles in /content:")
os.system("ls -lh /content")

# unzip
print("\nUnzipping...")
os.system('unzip /content/Pitt_Data.zip -d /content/pitt_data/')

# list extracted content
print("\nFiles extracted:")
os.system("ls -l /content/pitt_data/")


Mounted at /content/drive
Copying file...

Files in /content:

Unzipping...

Files extracted:


0

In [ ]:
# gettings pretrained downloaded files from the drive
from google.colab import drive
import os

# 1️⃣ Mount Google Drive
#drive.mount('/content/drive')

# 2️⃣ Paths to your model ZIPs in Drive
wav2vec_zip = "/content/drive/MyDrive/wav2vec2_model.zip"
bert_zip = "/content/drive/MyDrive/bert_model.zip"

# 3️⃣ Destination folders in Colab
wav2vec_folder = "/content/wav2vec/"
bert_folder = "/content/bert/"

# 4️⃣ Unzip models
os.system(f'unzip -o "{wav2vec_zip}" -d "{wav2vec_folder}"')
os.system(f'unzip -o "{bert_zip}" -d "{bert_folder}"')

# 5️⃣ Verify extracted files
print("\nContents of wav2vec folder:")
os.system(f'ls -l "{wav2vec_folder}"')

print("\nContents of BERT folder:")
os.system(f'ls -l "{bert_folder}"')


In [ ]:
#Main Mlp for classification. Just rin this thats all
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score
from google.colab import drive

# === Load combined embeddings ===
drive.mount('/content/drive')
combined_dict = torch.load("/content/drive/MyDrive/combined_new_embedding.pt")

# Convert dict values to tensors for dataset
features = torch.stack([v[0] for v in combined_dict.values()])  # [num_samples, 1536]
labels   = torch.stack([v[1] for v in combined_dict.values()])  # [num_samples]

# Create TensorDataset and DataLoader
dataset = TensorDataset(features, labels)

# Shuffle and split into train/test
num_samples = len(dataset)
train_size = int(0.8 * num_samples)
test_size = num_samples - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# === Define MLP ===
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.norm1 = nn.LayerNorm(256)
        self.drop1 = nn.Dropout(0.35)  # slightly higher dropout
        self.fc2 = nn.Linear(256, 64)
        self.norm2 = nn.LayerNorm(64)
        self.drop2 = nn.Dropout(0.25)  # slightly higher dropout
        self.fc3 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.norm1(out)
        out = self.relu(out)
        out = self.drop1(out)

        out = self.fc2(out)
        out = self.norm2(out)
        out = self.relu(out)
        out = self.drop2(out)

        out = self.fc3(out)
        return out

input_dim = features.shape[1]  # 1536
num_classes = 2
model = SimpleMLP(input_dim, num_classes)

# === Loss and optimizer ===
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-6)  # slightly lower lr

# === Training loop with early stopping ===
num_epochs = 90
best_accuracy = 0.0
best_model_state = None

for epoch in range(num_epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate on test set every epoch
    model.eval()
    with torch.no_grad():
        x_test = torch.stack([test_dataset[i][0] for i in range(len(test_dataset))])
        y_test = torch.tensor([test_dataset[i][1] for i in range(len(test_dataset))])
        outputs = model(x_test)
        _, predicted = torch.max(outputs, dim=1)
        accuracy = accuracy_score(y_test, predicted)

    # Save best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model_state = model.state_dict()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Test Acc: {accuracy*100:.2f}%')

# Load best model
model.load_state_dict(best_model_state)

print(f'\n✅ Best Test Accuracy Achieved: {best_accuracy*100:.2f}%')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch [10/90], Loss: 0.3142, Test Acc: 90.09%
Epoch [20/90], Loss: 0.3150, Test Acc: 90.99%
Epoch [30/90], Loss: 0.1901, Test Acc: 84.68%
Epoch [40/90], Loss: 0.2408, Test Acc: 87.39%
Epoch [50/90], Loss: 0.0647, Test Acc: 87.39%
Epoch [60/90], Loss: 0.1416, Test Acc: 89.19%
Epoch [70/90], Loss: 0.1029, Test Acc: 88.29%
Epoch [80/90], Loss: 0.0325, Test Acc: 82.88%
Epoch [90/90], Loss: 0.1815, Test Acc: 90.09%

✅ Best Test Accuracy Achieved: 91.89%


In [ ]:
# finetune_wav2vec_colab_safe.py
# code for fine-tuning wav2vec model
import os
# make allocator more flexible (set before heavy allocations)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128,expandable_segments:True"

import math
import torch
import random
import numpy as np
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import librosa
from sklearn.metrics import accuracy_score
from torch.amp import autocast, GradScaler

# -------- CONFIG --------
wav_folders = [
    "/content/pitt_data/Pitt_Data/Dementia/wav/cookie",
    "/content/pitt_data/Pitt_Data/Control/wav/cookie"
]
output_dir = "/content/drive/MyDrive/finetuned_wav2vec"
num_epochs = 5
batch_size = 8           # logical batch size
micro_batch_size = 1     # micro-batch size to fit GPU
learning_rate = 2e-5
seed = 42
sr = 16000
val_split = 0.15
# ------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

class AudioDataset(Dataset):
    def __init__(self, folders, processor, sr=16000):
        self.items = []
        for idx, folder in enumerate(folders):
            label = 1 if idx == 0 else 0
            if not os.path.exists(folder):
                print(f"⚠️ missing folder: {folder}")
                continue
            for fn in sorted(os.listdir(folder)):
                if fn.lower().endswith(".wav"):
                    self.items.append((os.path.join(folder, fn), label, os.path.splitext(fn)[0]))
        self.processor = processor
        self.sr = sr

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label, base = self.items[idx]
        audio, _ = librosa.load(path, sr=self.sr)
        inputs = self.processor(audio, sampling_rate=self.sr, return_tensors="pt", padding=True)
        input_values = inputs["input_values"][0]
        attention_mask = inputs.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask[0]
        return {"input_values": input_values, "attention_mask": attention_mask, "label": label, "basename": base}

def collate_fn(batch):
    input_values = [b["input_values"] for b in batch]
    attention_mask = [b["attention_mask"] for b in batch] if batch[0]["attention_mask"] is not None else None
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    basenames = [b["basename"] for b in batch]
    processed = processor.pad({"input_values": [iv for iv in input_values]}, padding=True, return_tensors="pt")
    if attention_mask is not None:
        processed = processor.pad({"input_values": [iv for iv in input_values],
                                   "attention_mask": [am for am in attention_mask]},
                                  padding=True, return_tensors="pt")
    processed["labels"] = labels
    return processed, basenames

# Load processor and model
processor = Wav2Vec2Processor.from_pretrained("/content/wav2vec/wav2vec2_model")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "/content/wav2vec/wav2vec2_model",
    num_labels=2,
    problem_type="single_label_classification"
)

# Enable gradient checkpointing to save memory
try:
    model.gradient_checkpointing_enable()
except Exception:
    pass

model.to(device)

# Prepare dataset and dataloaders
dataset = AudioDataset(wav_folders, processor, sr=sr)
n_total = len(dataset)
print(f"Total audio samples: {n_total}")
val_size = int(val_split * n_total)
train_size = n_total - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=lambda b: collate_fn([x for x in b]))
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=lambda b: collate_fn([x for x in b]))

optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.06 * total_steps), num_training_steps=total_steps)

# Mixed precision scaler
scaler = GradScaler()
best_val_acc = 0.0

def split_batch_dict(batch_dict, micro_bs):
    """Yield smaller batch dicts split along dim=0 for each tensor."""
    bsize = batch_dict["input_values"].shape[0]
    for i in range(0, bsize, micro_bs):
        small = {}
        for k, v in batch_dict.items():
            small[k] = v[i:i+micro_bs]
        yield small

# ----------------- Training Loop -----------------
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for processed_batch, _ in train_loader:
        micro_batches = list(split_batch_dict(processed_batch, micro_batch_size))
        total_samples_in_logical_batch = sum(mb["labels"].shape[0] for mb in micro_batches)
        accumulation_steps = max(1, math.ceil(total_samples_in_logical_batch / micro_batch_size))

        optimizer.zero_grad()
        batch_loss_sum = 0.0
        for mb in micro_batches:
            input_values = mb["input_values"].to(device)
            labels = mb["labels"].to(device)
            attention_mask = mb.get("attention_mask")
            if attention_mask is not None:
                attention_mask = attention_mask.to(device)

            with autocast("cuda"):
                outputs = model(input_values=input_values, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

            mb_size = labels.shape[0]
            batch_loss_sum += loss.item() * mb_size

            scaler.scale(loss / accumulation_steps).backward()

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

        logical_batch_loss = batch_loss_sum / max(1, total_samples_in_logical_batch)
        running_loss += logical_batch_loss

    avg_loss = running_loss / max(1, len(train_loader))

    # ----------------- Validation -----------------
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for processed_batch, basenames in val_loader:
            for mb in split_batch_dict(processed_batch, micro_batch_size):
                input_values = mb["input_values"].to(device)
                labels = mb["labels"].to(device)
                attention_mask = mb.get("attention_mask")
                if attention_mask is not None:
                    attention_mask = attention_mask.to(device)

                with autocast("cuda"):
                    outputs = model(input_values=input_values, attention_mask=attention_mask)
                    logits = outputs.logits
                preds = torch.argmax(logits, dim=-1)
                all_preds.extend(preds.cpu().numpy().tolist())
                all_labels.extend(labels.cpu().numpy().tolist())

    val_acc = accuracy_score(all_labels, all_preds) if len(all_labels) > 0 else 0.0
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f} - Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs(output_dir, exist_ok=True)
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)
        print(f"Saved best model to {output_dir} (val_acc={best_val_acc:.4f})")

    # Free GPU cache
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

print("Training finished. Best val acc:", best_val_acc)


In [ ]:
# finetune_bert_colab.py
# code for finetuing bert model
import os
import torch
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score

# -------- CONFIG --------
txt_folders = [
    "/content/pitt_data/Pitt_Data/Dementia/transcripts/cookie",
    "/content/pitt_data/Pitt_Data/Control/transcripts/cookie"
]
output_dir = "/content/drive/MyDrive/finetuned_bert"
num_epochs = 5
batch_size = 16
learning_rate = 2e-5
seed = 42
val_split = 0.15
max_length = 512
# ------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

class TextDataset(Dataset):
    def __init__(self, folders, tokenizer, max_length=512):
        self.items = []
        for idx, folder in enumerate(folders):
            label = 1 if idx == 0 else 0
            if not os.path.exists(folder):
                print(f"⚠️ missing folder: {folder}")
                continue
            for fn in sorted(os.listdir(folder)):
                if fn.lower().endswith(".txt"):
                    path = os.path.join(folder, fn)
                    with open(path, "r", encoding="utf-8") as f:
                        txt = f.read().strip()
                    if len(txt) == 0:
                        continue
                    self.items.append((txt, label, os.path.splitext(fn)[0]))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        txt, label, base = self.items[idx]
        enc = self.tokenizer(txt, truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]
        return {"input_ids": input_ids, "attention_mask": attention_mask, "label": label, "basename": base}

def collate_fn(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    basenames = [b["basename"] for b in batch]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}, basenames

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained("/content/bert/bert_model")
model = BertForSequenceClassification.from_pretrained("/content/bert/bert_model", num_labels=2)
model.to(device)

dataset = TextDataset(txt_folders, tokenizer, max_length=max_length)
n_total = len(dataset)
print(f"Total text samples: {n_total}")
val_size = int(val_split * n_total)
train_size = n_total - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=lambda b: collate_fn([x for x in b]))
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=lambda b: collate_fn([x for x in b]))

optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.06 * total_steps), num_training_steps=total_steps)

best_val_acc = 0.0
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch, _ in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()

    avg_loss = running_loss / max(1, len(train_loader))

    # validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch, _ in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    val_acc = accuracy_score(all_labels, all_preds) if len(all_labels) > 0 else 0.0
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs(output_dir, exist_ok=True)
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        print(f"Saved best model to {output_dir} (val_acc={best_val_acc:.4f})")

print("Training finished. Best val acc:", best_val_acc)
